# Faruq-v3 — Zhao STB Classification Screening (STB1, Seed 42)

Classification-path transfer of the Swin Transformer Block from Zhao et al., PR Letters 2025. Paper settings retained: window size 4, four heads, W-MSA followed by shifted W-MSA. Native YOLO26 localization uses untouched features. Transfer boundary: YOLO26 P3/P4/P5 replaces the S2A-Net ACL-aligned coarse-box feature, and a zero-initialized scalar gate is added only to preserve an exact D0 starting function. Test remains locked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/gds-stb-classification-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
cmd=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    r=subprocess.run(cmd)
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('clone gagal')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO}[dev]'],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_stb.py'],check=True)
print('PRE-FLIGHT PASS')


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan GPU Colab.'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT=Path('/content/faruq-development-v3-grouped'); GROUPED=DATA_ROOT/'faruq_grouped_summary.json'
if not GROUPED.is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert GROUPED.is_file(); assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-zhao-stb-screening-v1'
print(torch.cuda.get_device_name(0)); print(OUTPUT)


In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_stb_screening',
 '--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),'--control-summary',str(CONTROL),
 '--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
process=subprocess.Popen(command,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
for line in process.stdout: print(line,end='',flush=True)
rc=process.wait()
if rc: raise RuntimeError(f'STB1 gagal rc={rc}')


In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/stb_seed42_screening.json'
result=json.loads(SUMMARY.read_text())
assert result['evaluation_split']=='val' and result['test_images_accessed'] is False and result['test_opened'] is False
rows=[]
for name,v in result['controls'].items(): rows.append({'model':name,**v,'decision':'CONTROL'})
rows.append({'model':'STB1',**result['candidate']['STB1'],'decision':result['decision']})
display(pd.DataFrame(rows).style.format({k:'{:.2%}' for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('delta vs D0FT',result['delta_vs_D0FT']); print('delta vs ACMC1',result['delta_vs_ACMC1'])
print(result['design_boundary']); print('SUMMARY',SUMMARY)
print('Test tetap terkunci; RETAIN hanya discovery signal.')
